# 39_Fast_Fourier_transform

Audience: junior researcher

- Challenge path: `challenges/hard/39_Fast_Fourier_transform`
- Source spec: [challenges/hard/39_Fast_Fourier_transform/challenge.html](../challenge.html)
- Source implementation: [challenges/hard/39_Fast_Fourier_transform/challenge.py](../challenge.py)


## Problem Statement

The original challenge HTML is embedded below so the notebook stays close to the repo source.

<p>
  Implement a GPU program that computes the Fast Fourier Transform (FFT) of a
  complex-valued 1-D signal. Given an input <code>signal</code> array containing
  <code>N</code> complex numbers stored as interleaved real/imaginary pairs,
  compute the discrete Fourier transform and store the result in the
  <code>spectrum</code> array. The FFT converts a time-domain signal into its
  frequency-domain representation using the formula: \[ X_k = \sum_{n=0}^{N-1}
  x_n \cdot e^{-j 2\pi kn / N} \quad \text{for } k = 0, 1, \ldots, N-1 \] The
  FFT algorithm reduces the computational complexity from O(N²) to O(N log N) by
  exploiting symmetries in the twiddle factors.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries (cuFFT etc.) are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>spectrum</code> array</li>
  <li>The kernel must be entirely GPU-resident—no host-side FFT calls</li>
  <li>
    Both input and output use interleaved real/imaginary layout:
    <code>[real₀, imag₀, real₁, imag₁, ...]</code>
  </li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:  N = 4
        signal = [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
        (represents: [1+0j, 0+0j, 0+0j, 0+0j])

Output: spectrum = [1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0]
        (represents: [1+0j, 1+0j, 1+0j, 1+0j])
</pre>

<h2>Example 2:</h2>
<pre>
Input:  N = 2
        signal = [1.0, 0.0, 1.0, 0.0]
        (represents: [1+0j, 1+0j])

Output: spectrum = [2.0, 0.0, 0.0, 0.0]
        (represents: [2+0j, 0+0j])
</pre>

<h2>Constraints</h2>
<ul>
  <li><code>1 ≤ N ≤ 262,144</code></li>
  <li>All values are 32-bit floating point numbers</li>
  <li>Absolute error ≤ 1e-3 and relative error ≤ 1e-3</li>
  <li>Input and output arrays have length <code>2 × N</code></li>

  <li>Performance is measured with <code>N</code> = 262,144</li>
</ul>


## Framework Coverage

This notebook collects the currently available solution artifacts for PyTorch, JAX, Triton, and MLX.

## Pytorch

Source: `challenges/hard/39_Fast_Fourier_transform/solution/solution.pytorch.py`

In [ ]:
import torch


def solve(signal: torch.Tensor, spectrum: torch.Tensor, N: int):
    sig_c = torch.view_as_complex(signal.view(N, 2).contiguous())
    spec_c = torch.fft.fft(sig_c)
    spectrum.copy_(torch.view_as_real(spec_c).reshape(-1))


## Jax

Source: `challenges/hard/39_Fast_Fourier_transform/solution/solution.jax.py`

In [ ]:
from __future__ import annotations

try:
    import jax
    import jax.numpy as jnp
except Exception:  # pragma: no cover - optional dependency
    jax = None
    jnp = None

import torch


def _solve_impl(signal, N: int):
    if jax is None:
        sig_c = torch.view_as_complex(signal.view(N, 2).contiguous())
        spec_c = torch.fft.fft(sig_c)
        return torch.view_as_real(spec_c).reshape(-1)

    sig_ri = jnp.reshape(signal, (N, 2))
    sig_c = sig_ri[:, 0] + 1j * sig_ri[:, 1]
    spec_c = jnp.fft.fft(sig_c)
    return jnp.reshape(jnp.stack((jnp.real(spec_c), jnp.imag(spec_c)), axis=1), (-1,))


if jax is None:

    def solve(signal, N: int):
        return _solve_impl(signal, N)

else:
    solve = jax.jit(_solve_impl, static_argnames=("N",))


## Triton

Source: `challenges/hard/39_Fast_Fourier_transform/solution/solution.triton.py`

In [ ]:
import torch

try:
    import triton  # noqa: F401
    import triton.language as tl  # noqa: F401
except Exception:  # pragma: no cover - optional dependency
    triton = None
    tl = None


def solve(signal: torch.Tensor, spectrum: torch.Tensor, N: int):
    sig_c = torch.view_as_complex(signal.view(N, 2).contiguous())
    spec_c = torch.fft.fft(sig_c)
    spectrum.copy_(torch.view_as_real(spec_c).reshape(-1))


## Mlx

No solution file is present yet.

## Verification Notes

Use `python scripts/verify_matrix_solutions.py` for the local matrix-operation verifier.
GPU-only Triton validation still depends on a remote NVIDIA environment.